# §1. Feature Engineering

In [1]:
!python --version

Python 3.13.11


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATAPATH = Path("../data")


In [3]:
def read_data(filename, datapath=DATAPATH):
    file_path = datapath / filename
    if file_path.is_file():
        return pd.read_csv(file_path)
    else:
        raise FileNotFoundError(f"{file_path} does not exist.")
    
train = read_data("train.csv")
test = read_data("test.csv")
chief_complaints = read_data("chief_complaints.csv")
patient_history = read_data("patient_history.csv")

## §1.1 Chief Complaint Analysis

In [4]:
chief_complaints.head()

,patient_id,chief_complaint_raw,chief_complaint_system
0,TG-UXRGA9UCO,"thunderclap headache, worsening with movement",neurological
1,TG-B19DBBS2G,"contraception advice, intermittent",genitourinary
2,TG-GZ97W7M6V,"general health question, intermittent",other
3,TG-THIB2TN9Q,"erythema migrans tick bite, intermittent",dermatological
4,TG-J3U3LQ2QY,"cellulitis localised, intermittent",dermatological


In [5]:
DROP_COLUMNS = ['disposition', 'ed_los_hours']

train = train.merge(chief_complaints[['patient_id', 'chief_complaint_raw']], on="patient_id", how="left")
train = train.drop(columns=DROP_COLUMNS)
test = test.merge(chief_complaints[['patient_id', 'chief_complaint_raw']], on="patient_id", how="left")

assert "chief_complaint_raw" in train.columns
assert "chief_complaint_raw" in test.columns

In [6]:
# number of distinct diagnoses for the patient by system categories
complaints = train.groupby('chief_complaint_system')['chief_complaint_raw']

complaints_count = complaints.nunique()
print(complaints_count)
print("")
# see three examples for complaint system:
for system in complaints_count.index:
    print(f"Examples for {system} complaints:")
    print(complaints.get_group(system).head(3).to_list())
    print("")


chief_complaint_system
ENT                 354
cardiovascular      359
dermatological      356
endocrine           359
gastrointestinal    360
genitourinary       342
infectious          358
musculoskeletal     357
neurological        353
ophthalmic          325
other               358
psychiatric         353
respiratory         356
trauma              359
Name: chief_complaint_raw, dtype: int64

Examples for ENT complaints:
['mild epistaxis controlled', 'ear pain with fever', 'mild epistaxis controlled']

Examples for cardiovascular complaints:
['palpitations with near-syncope, intermittent', 'ECG result inquiry with vomiting', 'antihypertensive refill request, since yesterday']

Examples for dermatological complaints:
['erythema migrans tick bite, intermittent', 'cellulitis localised, intermittent', 'shingles with pain with associated nausea']

Examples for endocrine complaints:
['Cushing features query，worsening over hours', 'hyperglycaemia with symptoms, no prior history of this', 

## § 1.1.2 Illness History of Patients


In [7]:
patient_history.head()

,patient_id,hx_hypertension,hx_diabetes_type2,hx_diabetes_type1,hx_asthma,hx_copd,hx_heart_failure,hx_atrial_fibrillation,hx_ckd,hx_liver_disease,...,hx_hypothyroidism,hx_hyperthyroidism,hx_hiv,hx_coagulopathy,hx_immunosuppressed,hx_pregnant,hx_substance_use_disorder,hx_coronary_artery_disease,hx_stroke_prior,hx_peripheral_vascular_disease
0,TG-UXRGA9UCO,1,0,0,0,0,1,0,0,0,...,1,1,0,0,1,0,0,0,0,0
1,TG-B19DBBS2G,0,0,0,1,0,1,0,0,1,...,1,0,0,0,1,0,1,1,0,0
2,TG-GZ97W7M6V,1,1,1,1,0,1,1,1,0,...,0,1,1,0,0,0,0,1,1,1
3,TG-THIB2TN9Q,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,1
4,TG-J3U3LQ2QY,1,1,0,0,1,1,1,0,1,...,1,1,0,1,1,0,1,1,1,0


In [8]:
patient_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype
---  ------                          --------------   -----
 0   patient_id                      100000 non-null  str  
 1   hx_hypertension                 100000 non-null  int64
 2   hx_diabetes_type2               100000 non-null  int64
 3   hx_diabetes_type1               100000 non-null  int64
 4   hx_asthma                       100000 non-null  int64
 5   hx_copd                         100000 non-null  int64
 6   hx_heart_failure                100000 non-null  int64
 7   hx_atrial_fibrillation          100000 non-null  int64
 8   hx_ckd                          100000 non-null  int64
 9   hx_liver_disease                100000 non-null  int64
 10  hx_malignancy                   100000 non-null  int64
 11  hx_obesity                      100000 non-null  int64
 12  hx_depression                   100000 non-null  int64
 

In [9]:
train = train.merge(patient_history, on="patient_id", how="left")
test = test.merge(patient_history, on="patient_id", how="left")

## §1.2 Feature Selection Rationale

In [10]:
train.columns

Index(['patient_id', 'site_id', 'triage_nurse_id', 'arrival_mode',
       'arrival_hour', 'arrival_day', 'arrival_month', 'arrival_season',
       'shift', 'age', 'age_group', 'sex', 'language', 'insurance_type',
       'transport_origin', 'pain_location', 'mental_status_triage',
       'chief_complaint_system', 'num_prior_ed_visits_12m',
       'num_prior_admissions_12m', 'num_active_medications',
       'num_comorbidities', 'systolic_bp', 'diastolic_bp',
       'mean_arterial_pressure', 'pulse_pressure', 'heart_rate',
       'respiratory_rate', 'temperature_c', 'spo2', 'gcs_total', 'pain_score',
       'weight_kg', 'height_cm', 'bmi', 'shock_index', 'news2_score',
       'triage_acuity', 'chief_complaint_raw', 'hx_hypertension',
       'hx_diabetes_type2', 'hx_diabetes_type1', 'hx_asthma', 'hx_copd',
       'hx_heart_failure', 'hx_atrial_fibrillation', 'hx_ckd',
       'hx_liver_disease', 'hx_malignancy', 'hx_obesity', 'hx_depression',
       'hx_anxiety', 'hx_dementia', 'hx_epilepsy

In [11]:
train[['systolic_bp', 'heart_rate', 'shock_index']].head()

,systolic_bp,heart_rate,shock_index
0,79.0,57.3,0.725
1,131.7,97.3,0.739
2,94.7,75.6,0.798
3,134.2,109.0,0.812
4,140.1,113.7,0.812


In [12]:
# These are some derived features that can be calculated from the raw data:

# train.head().apply(lambda row: row['diastolic_bp'] + 1/3*(row['systolic_bp'] - row['diastolic_bp']) , axis=1) #mean arterial pressure
# train.head().apply(lambda row: (row['systolic_bp'] - row['diastolic_bp']) , axis=1) #pulse pressure
# train.head().apply(lambda row: (row['weight_kg'] / row['height_cm']**2)*1e4 , axis=1) #bmi
# train.head().apply(lambda row: (row['heart_rate'] / row['systolic_bp']) , axis=1) #shock index

In [13]:
id_variables = ['patient_id','site_id','triage_nurse_id']
physiological_variables = ['age', 'weight_kg', 'height_cm']
vital_signs_variables = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2']
other_variables = ['num_active_medications', 'num_comorbidities',  'gcs_total', 'pain_score', 'news2_score']
concurrency_variables = ['num_prior_ed_visits_12m', 'num_prior_admissions_12m']

categorical_variables = ['sex', 'arrival_mode', 'mental_status_triage', 'pain_location', 'chief_complaint_system', 'transport_origin']
text_variables = ['chief_complaint_raw']

hx_variables = ['hx_hypertension', 'hx_diabetes_type2',
       'hx_diabetes_type1', 'hx_asthma', 'hx_copd', 'hx_heart_failure',
       'hx_atrial_fibrillation', 'hx_ckd', 'hx_liver_disease', 'hx_malignancy',
       'hx_obesity', 'hx_depression', 'hx_anxiety', 'hx_dementia',
       'hx_epilepsy', 'hx_hypothyroidism', 'hx_hyperthyroidism', 'hx_hiv',
       'hx_coagulopathy', 'hx_immunosuppressed', 'hx_pregnant',
       'hx_substance_use_disorder', 'hx_coronary_artery_disease',
       'hx_stroke_prior', 'hx_peripheral_vascular_disease']


FEATURES = [
    *id_variables,
    *physiological_variables,
    *vital_signs_variables,
    *other_variables,
    *concurrency_variables,
    *categorical_variables,
    *text_variables,
    *hx_variables
]

LABEL = 'triage_acuity'

In [14]:
train = train[FEATURES + [LABEL]]
test = test[FEATURES]

A complete non-biased ML-based Triage system should consider only variable related to the patient's clinical condition and avoid variables that could introduce bias based on demographic or socio-economic factors.

For this reason, features related to demographic or socio-economic factors should be excluded from the model to minimize potential bias.
Some variables will be ignored since they are derived from other features or are not directly related to the patient's clinical condition.

Using [NEWS2 score](https://pmc.ncbi.nlm.nih.gov/articles/PMC9761428/) can help to assess the severity of patient's clinical condition. So it should be considered as an important feature in the model. Even though it is derived from vital signs (systolic_bp, diastolic_bp, heart_rate, respiratory_rate, temperature_c, spo2), the data misses information about Hypercapnic respiratory failure and whether the patient has supplemental oxygen, which are important for accurately calculating the NEWS2 score. See more [here](https://www.mdcalc.com/calc/10083/national-early-warning-score-news-2).

After predicting the ESI level, it could be used to guide clinical decision-making and resource allocation in the emergency department.
A possible further step could involve integrating the predicted ESI level with other clinical decision support tools (eg., ´disposition´, and Electronic Health Record data) to optimize patient care and resource management in the emergency department, as ´ed_los_hours´ (emergency department length of stay) of patients.

## § 1.3 Save data



In [18]:
print("Saving train and testprocessed data...")
train.to_csv(DATAPATH/'processed_train.csv', index=False)
test.to_csv(DATAPATH/'processed_test.csv', index=False)
print("")
print("Training Data")
print("Table with {} columns".format(train.shape[1]))
print("Table with {} rows".format(train.shape[0]))
print("Testing Data")
print("Table with {} columns".format(test.shape[1]))
print("Table with {} rows".format(test.shape[0]))
print("\nProcessed data saved.")

Saving train and testprocessed data...

Training Data
Table with 52 columns
Table with 80000 rows
Testing Data
Table with 51 columns
Table with 20000 rows

Processed data saved.
